# Light Rider Quantum Quickstart

Submit a quantum circuit to a real IQM backend using your Light Rider API key. Get your key from `/settings/keys` on the platform.

In [ ]:
import sys

major, minor = sys.version_info[:2]
print(f"Python {major}.{minor} detected.")
if major != 3 or minor < 10:
    print("⚠️ This notebook requires Python 3.10 or above. In Colab: Runtime → Change runtime type.")

In [ ]:
import requests

#@markdown Paste your Light Rider API key below, then run this cell.
api_key = "" #@param {type:"string"}
base_url = "https://platform.lightriderinc.com"

if not api_key:
    raise ValueError("Please enter your Light Rider API key above before running this cell.")

Build a simple Bell-pair circuit

Valid `backend` values:

- **Mock (free, unlimited):** `iqm-garnet-mock`, `iqm-emerald-mock`, `iqm-sirius-mock`
- **Real (costs credits, requires a prior purchase):** `iqm-garnet`, `iqm-emerald`, `iqm-sirius`

In [ ]:
backend = "iqm-garnet-mock"  # change this to the backend you want to use

circuit = {
    "num_qubits": 2,
    "instructions": [
        {"name": "h", "qubits": [0]},
        {"name": "cx", "qubits": [0, 1]},
        {"name": "measure", "qubits": [0], "clbits": [0]},
        {"name": "measure", "qubits": [1], "clbits": [1]},
    ],
}

response = requests.post(
    f"{base_url}/api/lr/quantum/submit",
    headers={"Authorization": f"Bearer {api_key}"},
    json={"backend": backend, "circuit": circuit, "shots": 1000},
)
response.raise_for_status()
job = response.json()
print("Job submitted:", job["job_uuid"])
print("Status:", job["status"])

In [ ]:
import time

job_id = job["job_uuid"]

while True:
    status_response = requests.get(
        f"{base_url}/api/lr/quantum/jobs/{job_id}",
        headers={"Authorization": f"Bearer {api_key}"},
    )
    status_response.raise_for_status()
    status_data = status_response.json()
    print("Status:", status_data["status"])

    if status_data.get("isInTerminalState"):
        break
    time.sleep(1)

result_response = requests.get(
    f"{base_url}/api/lr/quantum/jobs/{job_id}/result",
    headers={"Authorization": f"Bearer {api_key}"},
)
result_response.raise_for_status()
print("Result:", result_response.json())

In [ ]:
import matplotlib.pyplot as plt

counts = result_response.json()["counts"]
plt.bar(counts.keys(), counts.values())
plt.xlabel("Measurement outcome")
plt.ylabel("Count")
plt.title("Bell pair measurement results")
plt.show()

This circuit creates a **Bell pair** — two qubits entangled so they always agree when measured. You should see only `00` and `11` in the results, never `01` or `10`, roughly split 50/50. That agreement pattern is the signature of real quantum entanglement.

Every new signup gets 10 free Light Rider tokens automatically — no payment required. Mock backends (`iqm-garnet-mock`, `iqm-emerald-mock`, `iqm-sirius-mock`) are always free and unlimited regardless of your balance.

Switch `backend` to `iqm-garnet`, `iqm-emerald`, or `iqm-sirius` to run on real quantum hardware — that costs tokens, and requires having purchased credits at least once (the free signup tokens alone don't unlock real hardware). If you haven't purchased yet, you'll get a 402 (`purchase_required`) instead of a job id. If you have purchased before but your balance has since run out, you'll get a 402 (`insufficient_credits`) instead. Buy credits at `/settings/purchases/quantum-compute`.